# Phase 4 · Connect One Live GitHub Repository

**Owner: Karthik** · Board issue [#5](https://github.com/sulugambari/ai-agent-project/issues/5) · Course text `04-connected-rag-and-agent.md` Phase 4

**Read `HANDOVER.md` first** — sections 4 (findings), 6 (non-negotiables) and 8 (your brief).

## Why this notebook exists separately

`.ipynb` files are JSON with embedded outputs, so two people editing one notebook
produces merge conflicts that are painful to resolve. This is your space;
`northstar_build.ipynb` is Sulu's spine. Step 10.3 splices them for the final
documentation.

The first three cells are **copied verbatim** from the spine so both notebooks share the
same bootstrap, paths and chart styling. Don't change them here — if something needs
fixing, fix it in the spine and re-copy, so the two stay identical.

## What Phase 4 must deliver

| Step | Deliverable |
| --- | --- |
| 4.1 | `.env` configured; token boundary confirmed |
| 4.2 | Live connector: pagination, explicit error handling, title-independent stable IDs, intentional access policy |
| 4.3 | Fallback + controlled-failure test; no fabricated freshness · *figure: live vs fallback field parity* |

## Constraints that are already decided — do not relitigate

1. **Repository:** `sulugambari/ai-agent-project`. **No token needed.** Leave
   `GITHUB_TOKEN` empty.
2. **`allowed_roles = {"engineering"}`** on live work items. An *intentional* policy, not
   "whatever the API allowed". API reachability is not employee authorization
   (`ACCESS_MATRIX.md`).
3. **Stable ID must not depend on the issue title** — use the issue number, keep
   `node_id` in metadata. Required by `04`.
4. **Record `source_freshness`** = `live` | `fallback` and `fetched_at` on every record.
   Never present fallback data as live freshness.
5. **A malformed API response must raise**, not degrade into a record with empty
   `allowed_roles` — that would be world-readable. The `probe()` harness below is your
   regression test.
6. **`html_url` is the only genuine deep link in the whole product.** Preserve it; every
   other source's citation resolves to the record, not the origin system.
7. `uv add httpx` if the connector imports it — `04` requires it as a direct dependency.
   **No GitHub SDK.**

## The consequence to plan around

Our live repo's issues are the **project-management** issues (Phases 0–10), not Atlas
issues. EVAL-012 expects `GH-142`/`GH-149`, which exist only in the local export. So
treat the live repo as an **additional** work-item source merged with the local export,
and satisfy EVAL-012 through **disclosure of fallback state** in both configurations.
Don't fabricate Atlas-shaped issues to make the case pass — that's a joint decision if we
want it.

## Shared setup

Copied verbatim from `northstar_build.ipynb`. Run these first.

In [1]:
# --- Bootstrap -------------------------------------------------------------
# WHAT: locate the repository root and make it the working directory.
# WHY:  the starter's functions default to *relative* paths, e.g.
#           answer_with_baseline(..., data_root=Path("data/raw"))
#           DATABASE_PATH = Path("data/database/company.db")
#       Those resolve against the current working directory, which for a
#       notebook is notebooks/ — so they would silently fail here. Rather than
#       thread explicit paths through every call (and drift from how app.py and
#       api.py actually run), we chdir to the repo root once. The notebook then
#       exercises the same code paths the real product uses.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():          # walk up from notebooks/
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Could not locate repository root (no pyproject.toml found)")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

# Canonical locations, defined once and reused by every later phase.
DATA_RAW    = REPO_ROOT / "data" / "raw"          # local source exports
DATA_DB     = REPO_ROOT / "data" / "database" / "company.db"
DATA_EVAL   = REPO_ROOT / "data" / "evaluation" / "cases.json"
DATA_GEN    = REPO_ROOT / "data" / "generated"    # git-ignored: our own outputs
DATA_INDEX  = REPO_ROOT / "data" / "index"        # git-ignored: Chroma store
DELIVERABLES = REPO_ROOT / "deliverables"
FIGURES = DELIVERABLES / "figures"   # tracked: slide-deck and report images
DATA_GEN.mkdir(parents=True, exist_ok=True)

print(f"repo root : {REPO_ROOT}")
print(f"cwd       : {Path.cwd()}")
print(f"python    : {sys.version.split()[0]}")

repo root : /home/sulu/Neuefisch_wsl/ai-agent-project
cwd       : /home/sulu/Neuefisch_wsl/ai-agent-project
python    : 3.13.13


In [2]:
# --- Library imports -------------------------------------------------------
# WHAT: import the third-party libraries and the project's own contracts.
# WHY:  `company_assistant` is importable because `uv sync` installs this
#       project into .venv (src layout, declared in pyproject.toml). Importing
#       the real models here means the notebook is type-checked against the same
#       contracts the API and Streamlit app use — if we drift, this cell breaks.
import altair as alt
import pandas as pd

from company_assistant.api import EMPLOYEES
from company_assistant.models import (
    Answer, Citation, CompanyDocument, EmployeeContext, SearchResult,
)

print(f"altair {alt.__version__} | pandas {pd.__version__}")
print(f"fictional employee profiles: {', '.join(EMPLOYEES)}")

altair 6.2.2 | pandas 3.0.5
fictional employee profiles: maya, leo, priya, omar


In [3]:
# --- Shared Altair theme ---------------------------------------------------
# NOTE: Altair 6 replaced `alt.themes.register` with `@alt.theme.register`.
#       The old API is deprecated and emits warnings, so we use the new one.
@alt.theme.register("northstar", enable=True)
def northstar_theme() -> alt.theme.ThemeConfig:
    """Consistent, readable styling for every chart in this project."""
    return alt.theme.ThemeConfig({
        "config": {
            "view":   {"stroke": "transparent", "continuousWidth": 520, "continuousHeight": 280},
            "axis":   {"labelFontSize": 11, "titleFontSize": 12, "grid": True,
                       "gridColor": "#E2E8F0", "domainColor": "#94A3B8",
                       "tickColor": "#94A3B8", "labelColor": "#334155",
                       "titleColor": "#172033"},
            "legend": {"labelFontSize": 11, "titleFontSize": 12, "labelColor": "#334155"},
            "title":  {"fontSize": 14, "anchor": "start", "color": "#172033",
                       "subtitleFontSize": 11, "subtitleColor": "#64748B"},
            "range":  {"category": ["#4677A8", "#3B8A5A", "#C86445", "#B77A1F",
                                    "#7A5AA8", "#5FA8A0"]},
        }
    })

# Semantic colours reused across phases so meaning stays stable chart to chart.
# Fixed here rather than per-chart: "denied" must look the same everywhere.
COLORS = {
    "allow":   "#3B8A5A",   # permitted / pass
    "deny":    "#B60205",   # forbidden / fail  (also = release blocker)
    "partial": "#B77A1F",   # partial / warning
    "neutral": "#64748B",   # not applicable
    "lexical": "#4677A8", "semantic": "#7A5AA8", "hybrid": "#3B8A5A",
}

def save_chart(chart: alt.Chart, name: str, *, caption: str | None = None) -> alt.Chart:
    """Persist a chart in two formats and return it for inline display.

    WHY TWO FORMATS — they serve different consumers:
      * Vega-Lite JSON -> data/generated/charts/  (git-ignored, regenerable)
        Consumed by the Phase 8 Streamlit dashboard, which renders Altair specs
        natively. Kept as a spec so it stays interactive and diff-friendly.
      * PNG @2x        -> deliverables/figures/   (tracked in git)
        Consumed by the final slide deck and the written deliverables. Tracked
        because a presentation asset must survive a clean checkout, and
        data/generated/ is git-ignored by design.

    `caption` is the one-line message the figure is meant to prove. It is
    recorded next to the file so the deck can be assembled from the ledger
    without re-deriving what each chart was for.
    """
    (DATA_GEN / "charts").mkdir(parents=True, exist_ok=True)
    FIGURES.mkdir(parents=True, exist_ok=True)
    chart.save(DATA_GEN / "charts" / f"{name}.json")
    chart.save(FIGURES / f"{name}.png", scale_factor=2.0)
    if caption:
        (FIGURES / f"{name}.txt").write_text(caption.strip() + "\n", encoding="utf-8")
    print(f"saved figure '{name}'  ->  deliverables/figures/{name}.png")
    return chart

## Regression harness — copied from step 3.1

`probe()` writes a malformed fixture into a temp directory and reports whether the
connector **raised** (acceptable — someone must fix the source) or was **silent**
(unacceptable — the record either enters the index unprotected or vanishes with no
signal that evidence is missing).

The four supplied connectors score **10 of 10 raised, 0 silent**. Your live connector
must clear the same bar. Write the equivalent probes for API responses: missing
`allowed_roles` assignment, absent `number`, absent `html_url`, malformed timestamp,
truncated pagination, HTTP 403/404/500, and a timeout.

In [4]:
# --- Malformed-record behaviour ------------------------------------------
# WHAT: feed each connector a deliberately broken record and record what happens.
# WHY:  the required evidence for step 3.1. Three outcomes are possible and only
#       two are acceptable:
#         RAISED  - loud failure. Acceptable: someone must fix the source.
#         DENIED  - parsed but excluded by permissions. Acceptable for access.
#         SILENT  - accepted, or dropped without complaint. NOT acceptable: the
#                   record either enters the index unprotected, or vanishes with
#                   no signal that evidence is missing.
import json
import shutil
import tempfile
from datetime import datetime, timezone

from company_assistant.connectors import (
    load_documents, load_emails, load_github_issues, load_slack_messages)

def probe(name, writer, loader):
    """Write a malformed fixture into a temp dir and report the connector's behaviour."""
    with tempfile.TemporaryDirectory() as tmp:
        folder = Path(tmp)
        writer(folder)
        try:
            loaded = loader(folder)
        except Exception as exc:
            return {"case": name, "outcome": "RAISED",
                    "detail": f"{type(exc).__name__}: {str(exc)[:80]}"}
        return {"case": name, "outcome": "SILENT",
                "detail": f"accepted or dropped without error; {len(loaded)} record(s) returned"}

SLACK_OK = {"source_id": "SLACK-T-1", "channel": "t", "author": "a",
            "timestamp": "2026-08-01T00:00:00+00:00", "text": "body",
            "allowed_roles": ["engineering"]}

def w_slack(mutate):
    def writer(folder):
        rec = {**SLACK_OK, **mutate}
        (folder / "t.json").write_text(json.dumps([rec]), encoding="utf-8")
    return writer

def w_doc(front):
    def writer(folder):
        (folder / "t.md").write_text(f"---\n{front}\n---\n\nbody\n", encoding="utf-8")
    return writer

def w_email(headers):
    def writer(folder):
        (folder / "t.eml").write_text(
            headers + '\nContent-Type: text/plain; charset="utf-8"\n\nbody\n', encoding="utf-8")
    return writer

def w_gh(mutate):
    def writer(folder):
        rec = {"source_id": "GH-T-1", "number": 1, "title": "t", "body": "b",
               "state": "open", "author": "a", "updated_at": "2026-08-01T00:00:00+00:00",
               "allowed_roles": ["engineering"], **mutate}
        (folder / "t.json").write_text(json.dumps([rec]), encoding="utf-8")
    return writer

results = [
    probe("Slack: allowed_roles missing",       w_slack({"allowed_roles": None}),        load_slack_messages),
    probe("Slack: allowed_roles empty list",    w_slack({"allowed_roles": []}),          load_slack_messages),
    probe("Slack: unknown role name",           w_slack({"allowed_roles": ["exec"]}),    load_slack_messages),
    probe("Slack: source_id missing",           w_slack({"source_id": None}),            load_slack_messages),
    probe("Document: allowed_roles absent",     w_doc("source_id: D-1\ntitle: t\neffective_at: 2026-08-01T00:00:00+00:00"), load_documents),
    probe("Document: bad confidentiality",      w_doc("source_id: D-1\ntitle: t\neffective_at: 2026-08-01T00:00:00+00:00\nconfidentiality: public\nallowed_roles:\n  - engineering"), load_documents),
    probe("Email: X-Access-Roles missing",      w_email("From: a@b.c\nSubject: t\nX-Source-ID: E-1\nX-Occurred-At: 2026-08-01T00:00:00+00:00"), load_emails),
    probe("Email: X-Source-ID missing",         w_email("From: a@b.c\nSubject: t\nX-Access-Roles: engineering\nX-Occurred-At: 2026-08-01T00:00:00+00:00"), load_emails),
    probe("GitHub: allowed_roles missing",      w_gh({"allowed_roles": None}),           load_github_issues),
    probe("GitHub: unknown role name",          w_gh({"allowed_roles": ["ops"]}),        load_github_issues),
]
malformed = pd.DataFrame(results)
display(malformed)

silent = malformed[malformed.outcome == "SILENT"]
print(f"\n{len(malformed)} malformed cases: "
      f"{(malformed.outcome == 'RAISED').sum()} raised, {len(silent)} silent")
assert silent.empty, f"silent failures found - evidence could disappear unnoticed:\n{silent}"
print("every malformed record fails LOUDLY at parse time - none is silently dropped or accepted")

,case,outcome,detail
0,Slack: allowed_roles missing,RAISED,ValidationError: 1 validation error for SlackM...
1,Slack: allowed_roles empty list,RAISED,ValueError: Source access metadata must contai...
2,Slack: unknown role name,RAISED,ValueError: Unknown employee roles: ['exec']
3,Slack: source_id missing,RAISED,ValidationError: 1 validation error for SlackM...
4,Document: allowed_roles absent,RAISED,ValidationError: 1 validation error for Docume...
5,Document: bad confidentiality,RAISED,ValidationError: 1 validation error for Docume...
6,Email: X-Access-Roles missing,RAISED,ValueError: /tmp/tmphp7p8ovo/t.eml is missing ...
7,Email: X-Source-ID missing,RAISED,ValueError: /tmp/tmpppxllong/t.eml is missing ...
8,GitHub: allowed_roles missing,RAISED,ValidationError: 1 validation error for GitHub...
9,GitHub: unknown role name,RAISED,ValueError: Unknown employee roles: ['ops']



10 malformed cases: 10 raised, 0 silent
every malformed record fails LOUDLY at parse time - none is silently dropped or accepted


## 4.1 · Configuration and the token boundary

*Your cells here.* Confirm `GITHUB_REPOSITORY` is read from configuration, that the code
path works with **no** token, and that no credential can reach a prompt, trace, indexed
record or figure.

In [5]:
# 4.1

## 4.2 · The live connector

*Your cells here.* Suggested order: fetch one page → handle pagination → handle errors
explicitly → normalize to `CompanyDocument` → apply the intentional access policy →
run the malformed-response probes.

In [6]:
# 4.2

## 4.3 · Fallback and controlled failure

*Your cells here.* Run once with the live source available and once with it deliberately
unavailable. Show that the fallback is disclosed and that freshness is never fabricated.

**Figure to produce:** live vs fallback field parity — which fields each path populates,
so a regression that drops `html_url` or `allowed_roles` is visible. Compare against
`3_1_citation_affordances.png`, which has the `github live (Phase 4)` row marked
*Planned*.

Use `save_chart(fig, "4_3_live_vs_fallback", caption="...")` so it reaches
`deliverables/figures/` and the slide-deck ledger automatically.

In [7]:
# 4.3

## When you finish

1. Tick steps 4.1–4.3 on board issue [#5](https://github.com/sulugambari/ai-agent-project/issues/5) and comment the findings.
2. Append presentable findings to `deliverables/SLIDE_DECK.md` (slide 8 is
   *"Connecting a live source safely"*).
3. Fill the **Live GitHub repository** row in the Source Governance table of
   `ACCESS_MATRIX.md` if your implementation differs from what step 2.2 assumed.
4. Record a `D-003` entry in `DECISIONS.md` for the live-source decision: rate-limit
   assumptions, fallback behaviour, and the token boundary. `04` asks for this explicitly.
5. Tell Sulu — this is handover **H1**, and Phase 5 indexes your records.